In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hw10.ipynb")

<div class="alert alert-success">

#### Homework 10 Supplemental Notebook
    
# Eigenvalues and Eigenvectors

### EECS 245, Winter 2026 at the University of Michigan
    
</div>

### Instructions

Most homeworks will have Jupyter Notebooks, like this one, designed to supplement the theoretical problems. 

To write and run code in this notebook, you have two options:

1. **Set up a Jupyter Notebook environment locally, and use `git` to clone our course repository (preferred).** For instructions on how to do this, see the [**Environment Setup**](https://eecs245.org/env-setup) page of the course website.
1. **Use the EECS 245 DataHub.** To do this, click the link provided in the Homework 10 PDF. Before doing so, read the instructions on the [**Environment Setup**](https://eecs245.org/env-setup/#option-2-using-the-eecs-245-datahub) page on how to use the DataHub.

There are two types of problems in this notebook:
- Problem 6f) doesn't involve any programming; instead, you just need to run all of the code we've already provided you for it, and provide answers and screenshots in your Homework 10 PDF.
- Problem 7 is **entirely** autograded. To receive credit for Problem 7, submit your completed notebook to the Homework 10, Problem 7 Code autograder on Gradescope. Your submission time for Homework 10 is the **latter** of your PDF and code submission times. This particular programming problem has **no hidden test cases** – the test cases on Gradescope are exactly the ones available in your notebook.

In [ ]:
# Run this cell.
import numpy as np
import pandas as pd
import time

pd.options.plotting.backend = "plotly"

import plotly.express as px
import plotly.io as pio
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

# Set default layout for all plotly figures
import plotly.graph_objs as go

custom_template = go.layout.Template(pio.templates["plotly_white"])
custom_template.layout.plot_bgcolor = "white"
custom_template.layout.paper_bgcolor = "white"
custom_template.layout.margin = dict(l=60, r=60, t=60, b=60)
custom_template.layout.width = 700
custom_template.layout.font = dict(
    family="Palatino Linotype, Palatino, serif",
    color="black"
)

pio.templates["custom"] = custom_template
pio.templates.default = "custom"

import matplotlib.pyplot as plt
import networkx as nx

## Problem 6f: Regularization Returns

---

As was introduced in the Homework 10 PDF, ridge regression is the problem of finding the vector that minimizes the objective function

$$R_\text{ridge}(\vec{w}) = \lVert \vec{y} - X \vec{w} \rVert^2 + \lambda \lVert \vec w \rVert^2$$

Once we find that vector, we can make predictions using $\vec{p} = X \vec{w}_\text{ridge}^*$, where $X$ is a design matrix with information about the individuals we want to make predictions for.

The vector that minimizes $R_\text{ridge}(\vec{w})$ is:

$$\vec{w}_\text{ridge}^* = (X^TX + \lambda I)^{-1} X^T \vec{y}$$

$\vec{w}_\text{ridge}^*$ is unique, whether or not $X$ is full rank, as we proved earlier in Problem 6.

This is different from $\vec{w}_\text{OLS}^* = (X^TX)^{-1}X^T \vec{y}$, which is only uniquely defined when $X^TX$ is invertible; otherwise, all of the infinitely many solutions to the normal equations, $X^TX \vec{w}_\text{OLS}^* = X^T \vec{y}$, minimize mean squared error. "OLS" refers to "ordinary least squares", the process of minimizing mean squared error, $R_\text{sq}(\vec{w}) = \frac{1}{n} \lVert \vec y - X \vec w \rVert_2^2$, with no regularization.

<hr style="border: none; border-top: 3px solid #4169E1; margin: 20px 0;">

Some lingering questions:
- **(Problem 6f)** Make sure to include your answers to both of the following question in you Homework 10 PDF:
    - Why is it called ridge regression? 
    - How does the inclusion of the $\lambda \lVert \vec w \rVert^2$ term change the **convexity** of the loss surface?
- **(Further reading, linked in PDF)** What should we set $\lambda$ to, and why are we doing this in the first place?

<hr style="border: none; border-top: 3px solid #4169E1; margin: 20px 0;">

Let's explore! There's a lot of code in this question, but it's all already written for you. Your job is to work through our exploration, and then provide screenshots and answer questions in your Homework 10 PDF.

Run the cell below to load in a dataset containing the weights and heights of 25,000 18 year olds."

In [ ]:
people = pd.read_csv('data/heights-weights.csv')
people.head()

Note that there are two height columns, `Height (inches)` and `Height (cm)`. Remember that 1 inch is equal to 2.54 centimeters.

Throughout this question, we'll aim to predict `Weight (Pounds)` using the other two features. Let's start by plotting `Weight (Pounds)` vs. `Height (Inches)`:

In [ ]:
fig = px.scatter(people, x='Height (Inches)', y='Weight (Pounds)', title='Weight vs. Height for 25,000 18 Year Olds')
fig.show()

And `Weight (Pounds)` vs. `Height (Inches)` and `Height (cm)`:

In [ ]:
fig = px.scatter_3d(people, x='Height (Inches)', y='Height (cm)', z='Weight (Pounds)', title='Weight vs. Height for 25,000 18 Year Olds')
fig.show()

Drag the plot around. Unsurprisingly, the data lies on a 2-dimensional plane in $\mathbb{R}^3$, since `Height (Inches)` and `Height (cm)` are just scalar multiples of each other.

As we've seen before, this means that when we go to train our model,

$$\text{predicted Weight (Pounds)}_i = w_0 + w_1 \cdot \text{Height (Inches)}_i + w_2 \cdot \text{Height (cm)}_i$$

and try to minimize (unregularized) mean squared error, we'll end up with infinitely many possible parameter vectors, $\vec w^*$. Our $n \times 3$ design matrix $X$ is not full rank: it has a rank of $2$. We know this, and `numpy` knows it too:

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(people[['Height (Inches)', 'Height (cm)']], 
                                                    people['Weight (Pounds)'],
                                                    random_state=98)

X_train_design = X_train.copy()
X_train_design['1'] = 1
X_train_design = X_train_design[['1', 'Height (Inches)', 'Height (cm)']]
X_train_design

In [ ]:
np.linalg.matrix_rank(X_train_design)

**So, how does ridge regression help with this problem?**

To illustrate, let's look at a graph of $R_\text{ridge}(\vec{w}) = \lVert \vec y - X \vec w \rVert^2 + \lambda \lVert \vec w \rVert^2$, for different values of $\lambda$. The graph you're about to see is also known as a _loss surface_.

Remember, our model is of the form:

$$\text{predicted Weight (Pounds)}_i = w_0 + w_1 \cdot \text{Height (Inches)}_i + w_2 \cdot \text{Height (cm)}_i$$

There are three axes in the graph that appears:

- One for different values of $w_1$.
- One for different values of $w_2$.
- One for the value of $R_\text{ridge}(-82.77, w_1, w_2)$. Since we have three parameters, we'd need four axes to truly visualize $R_\text{ridge}(\vec{w})$, so we've fixed a particular value of $w_0$. (The interesting part of the graph doesn't involve the intercept, anyways, since the linear dependence is between `Height (Inches)` and `Height (cm)`.)

Run the cell below and interact with the slider that appears!

<hr style="border: none; border-top: 3px solid #FF8C00; margin: 20px 0;">

In [ ]:
X_train_design = X_train.copy()
X_train_design['1'] = 1
X_train_design = X_train_design[['1', 'Height (Inches)', 'Height (cm)']]

def show_loss_surface(reg_lambda):
    def mse_for_weight_model(w):
        w = np.array([-82.77, w[0], w[1]])
        return np.mean((y_train - X_train_design @ w) ** 2) + reg_lambda * (w[0] ** 2 + w[1] ** 2)

    if reg_lambda > 0:
        identity = np.eye(X_train_design.shape[1])
        identity[0, 0] = 0
        w_star = np.linalg.solve(X_train_design.T @ X_train_design + X_train_design.shape[0] * reg_lambda * identity,
                                 X_train_design.T @ y_train)

    num_points = 25
    uvalues = np.linspace(-1.5, 2, num_points)
    vvalues = np.linspace(-2, 5, num_points)
    (u,v) = np.meshgrid(uvalues, vvalues)
    ws = np.vstack((u.flatten(), v.flatten()))
    MSE = np.array([mse_for_weight_model(w) for w in ws.T])
    loss_surface = go.Surface(x=u, y=v, z=np.reshape(MSE, u.shape), showscale=False)

    fig = go.Figure(data=[loss_surface])
    # fig.add_trace(opt_point)
    fig.update_layout(title=("Mean Squared Error" if reg_lambda == 0 else f"Ridge Regression with Lambda = {reg_lambda}"), scene = dict(
        xaxis_title = "w1: Height (Inches)",
        yaxis_title = "w2: Height (cm)"),
                     width=700, height=500, showlegend=False)
    fig.show()

from ipywidgets import interact
interact(show_loss_surface, reg_lambda=(0, 100000, 5000));

<hr style="border: none; border-top: 3px solid #FF8C00; margin: 20px 0;">

Some things to note:
- When you drag the `reg_lambda` slider to 0, and see the title "Mean Squared Error", you should notice a long "ridge" at the bottom of the loss surface! All values of $w_1$ and $w_2$ that fall on that ridge minimize mean squared error. **This ridge is problematic!**
- As you increase `reg_lambda`, the surface looks more and more like a bowl curved upwards, and less "ridgy". **Ridge regression removes the ridge!**
- As you increase `reg_lambda`, look at the $z$-axis of the resulting plot.

So, there, we have our answer! Ridge regression removes the "ridge" of infinitely many solutions when the design matrix isn't full rank. **Remember to include the relevant answers in your Homework 10 PDF for Problem 6f.**

## Problem 7: PageRank

---

In this part of the homework, you'll replicate the PageRank algorithm, the algorithm that Google uses to decide how to rank search results. Make sure you've already completed Problem 5 (on adjacency matrices) and read [Chapter 9.3](https://notes.eecs245.org/eigenvalues-and-eigenvectors/markov-chains-adjacency-matrices/) before proceeding.

We'll start by defining a 2D array, `A`, representing the adjacency matrix for some network of four pages.

In [ ]:
A = np.array([[0, 1 / 2, 1 / 2, 1 / 3],
              [1, 0, 0, 1 / 3],
              [0, 0, 0, 1 / 3],
              [0, 1 / 2, 1 / 2, 0]])
A

The function below draws a network given an adjacency matrix. Run the cell below to see a visualization of `A`'s network.

In [ ]:
def plot_from_adjacency(adjacency_matrix, node_sizes=0.25):
    np.random.seed(25)
    plt.figure(figsize=(8, 5))
    G = nx.from_numpy_array(adjacency_matrix.T, create_using=nx.DiGraph)
    layout = nx.spring_layout(G)
    labels_dict={i: i+1 for i in range(adjacency_matrix.shape[0])}
    nx.draw(G, layout, 
            node_size=15000 * node_sizes, labels=labels_dict, with_labels=True, font_color='white', font_weight='bold', font_size=15, 
            connectionstyle='arc3, rad = 0.1')
    plt.show()

plot_from_adjacency(A)

### Problem 7a)

Having to specify an adjacency matrix manually is slightly cumbersome. It is more natural and convenient for us to describe the links between webpages using a dictionary. Once such example is as follows:

In [ ]:
example_net = {
    1: [2],
    2: [1, 4],
    3: [1, 4],
    4: [1, 2, 3]
}

In the above "network dictionary", we are told that:
- Page 1 links to Page 2.
- Page 2 links to Pages 1 and 4.
- Page 3 links to Pages 1 and 4.
- Page 4 links to Pages 1, 2, and 3.

**Note that this dictionary describes the same network that the adjacency matrix `A` does.**

Below, complete the implementation of the function `create_adjacency`, which takes in a network dictionary, `network` (formatted similarly to `example_net`) and returns the adjacency matrix that corresponds to `network`. Example behavior is given below.

```python
>>> create_adjacency(example_net)
array([[0.        , 0.5       , 0.5       , 0.33333333],
       [1.        , 0.        , 0.        , 0.33333333],
       [0.        , 0.        , 0.        , 0.33333333],
       [0.        , 0.5       , 0.5       , 0.        ]])
```

A few notes:
- It is **not** guaranteed that there are 4 pages in the network. 
- It **is** guaranteed that all pages link to at least one page – potentially itself – and that there is at least one page.
- Remember that adjacency matrices are 1-indexed, like in math, but Python is 0-indexed.

Some guidance:
- Look into `np.zeros`.
- You can use (multiple) `for`-loops.

In [ ]:
def create_adjacency(network):
    ...

# Feel free to change this input to make sure your function works correctly.
create_adjacency(example_net)

In [ ]:
grader.check("p7a")

### Problem 7b)

Complete the implementation of the function `compute_scores`, which takes in an adjacency matrix `matrix` of shape `(n, n)`, where `n` is a positive integer, and returns an array of length `n` containing the PageRank scores of all pages. 

```python
# Remember, compute_scores should work for any adjacency matrix, not just A!
>>> compute_scores(A)
array([0.30769231, 0.38461538, 0.07692308, 0.23076923])
```

_Hint: Remember that the PageRank scores are stored in the eigenvector for a particular eigenvalue. Make sure that the scores in your vector add up to 1. While you can theoretically use `np.linalg.eig` to do this, it's slightly tricky to retrieve a specific eigenvector from its output since the eigenvalues it gives you aren't necessarily sorted. You could also use `np.linalg.matrix_power` with an exponent of 100._

In [ ]:
def compute_scores(matrix):
    ...

# Feel free to change this input to make sure your function works correctly.
compute_scores(A)

In [ ]:
grader.check("p7b")

We can change the sizes of the pages in our network to be proportional to their PageRank scores. To do this, use the `node_sizes` argument in `plot_from_adjacency`.

In [ ]:
plot_from_adjacency(A, node_sizes=compute_scores(A))

### Problem 7c)

Complete the implementation of the function `pagerank`, which takes in an adjacency matrix `matrix` of shape `(n, n)`, where `n` is a positive integer, and returns a list containing the **numbers** of the pages in the matrix, in **decreasing** order of PageRank score as computed by `compute_scores`. If there are ties in scores, return the pages in any order. Example behavior is given below.

```python
>>> pagerank(A)
[2, 1, 4, 3]
```

_Hint: Look into `np.argsort`. This should only take 2-3 lines; do not write a `for`-loop._

In [ ]:
def pagerank(matrix):
    ...

# Feel free to change this input to make sure your function works correctly.
pagerank(A)

In [ ]:
grader.check("p7c")

Once you've completed `pagerank`, run the following cell to visualize your work.

In [ ]:
# Here's another example network, which you can use to test your code.
test_net = {1: [6], 2: [1, 3, 4, 6], 3: [2, 5, 6], 4: [1], 5: [1, 2, 6], 6: [2, 4]}
print(test_net)
test_net_adjacency = create_adjacency(test_net)
test_net_scores = compute_scores(test_net_adjacency)
plot_from_adjacency(test_net_adjacency, node_sizes=test_net_scores)

In [ ]:
test_net_scores

Let's wrap up this problem up with a puzzle. Consider the following network:

In [ ]:
weird_net = {
    1: [1],
    2: [1, 3, 5],
    3: [2, 4],
    4: [1, 2, 3],
    5: [3]
}

Note that Page 1 links to itself and to no other pages. Practically speaking, we can interpret Page 1 as being a "dead end" or "sinkhole", with no outgoing links.

Run the cells below to compute the adjacency matrix and scores for the above network and to visualize it.

In [ ]:
weird_matrix = create_adjacency(weird_net)
weird_scores = compute_scores(weird_matrix)
print(weird_scores)
plot_from_adjacency(weird_matrix, node_sizes=weird_scores)

It appears that Page 1's score is 1, and all of the other pages' scores are 0! (`9.6409e-10` means $9.64 \cdot 10^{-10}$, which is a really, really small number.)

**Think** about the answers to the following two prompts. You don't have to write your answers anywhere, but you're expected to have thought about them (as they may appear on the Final Exam):

- Why do you think Page 1's score is so high, and the other pages' scores are so low? (How do we interpret the score of a page?)
- Read the [Damping factor](https://en.wikipedia.org/wiki/PageRank#Damping_factor) section of the Wikipedia article on PageRank. How would damping would prevent the score of Page 1 from becoming 1? How does the damping factor change the matrix whose eigenvalues we compute?

## Finish Line 🏁

To get credit for Problem 6f, make sure to include all relevant screenshots in your Homework 10 PDF. To get credit for the autograded coding problem, submit this notebook to Gradescope.
1. Select `Kernel -> Restart & Run All` to ensure that you have executed all cells, including the test cells.
2. Read through the notebook to make sure everything is fine and all public tests passed.
3. Run the cell below to run all tests, and make sure that they all pass.
4. Download your notebook using `File -> Download`, then upload your notebook to Gradescope under "Homework 10, Problem 7 Code".
5. Stick around for a few minutes while the Gradescope autograder grades your work. Make sure you see that all **public tests** have passed on Gradescope. **This particular problem has no hidden test cases, so you should see a full score shortly after submitting.**